# CellNiche tutorial: multi-slice spatial graph construction and training

This notebook demonstrates how to use **CellNiche** on a multi-slice / multi-sample spatial omics dataset.

The key difference from the single-slice workflow is graph construction:

- For a single slice, CellNiche can build a spatial graph directly from `adata.obsm["spatial"]` or `adata.obs[["x", "y"]]`.
- For multiple slices, directly building one graph from all 2D coordinates may create artificial edges across unrelated slices.
- Therefore, for multi-slice data, we first construct a **sample-aware spatial graph** using Squidpy with `library_key="sample"`.
- CellNiche then reads the precomputed graph from `adata.obsp["spatial_connectivities"]` by setting:
  ```yaml
  multi_slice: true
  connectivity_key: "spatial"
  ```

Expected input after preprocessing:

```python
adata_concat.obs["sample"]              # slice/sample ID
adata_concat.obsm["spatial"]            # 2D spatial coordinates
adata_concat.obsp["spatial_connectivities"]  # sample-aware spatial graph from Squidpy
```

In [1]:
! hostname

c01


In [2]:
# enable autoreload
%load_ext autoreload
%autoreload 2

In [3]:
import os
import time
import sys
import random
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import torch
from anndata import AnnData
import anndata
import seaborn as sns
import matplotlib.pyplot as plt
import anndata as ad

from sklearn import metrics
import multiprocessing as mp
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import scipy.sparse as sp
import scipy.linalg

import warnings
warnings.filterwarnings("ignore")

/share/home/liangzhongming/anaconda3/envs/cellniche/lib/python3.9/site-packages/cudf/utils/gpu_utils.py:62: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))
/share/home/liangzhongming/anaconda3/envs/cellniche/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/share/home/liangzhongming/anaconda3/envs/cellniche/lib/python3.9/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [4]:
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.set_figure_params(dpi=150,
                     dpi_save=300,
#                      facecolor='w',
#                      frameon=False, # frameon=True
#                      figsize=(4,4)
                    ) 
%config InlineBackend.figure_format='retina'
%matplotlib inline

scanpy==1.10.3 anndata==0.10.9 umap==0.5.9.post2 numpy==1.26.4 scipy==1.13.1 pandas==2.2.2 scikit-learn==1.6.1 statsmodels==0.14.5 igraph==0.11.9 pynndescent==0.5.13


In [5]:
# import cellniche as cn
sys.path.append('/share/home/liangzhongming/phd_code/ModelTest/CellNiche')
import cellniche as cn

## Define input slices

In this example, we integrate four representative mouse-brain slices from four atlas folders. Users should replace atlas_dirs and target_files with their own data paths and file names.

Each input `.h5ad` file contain:

- cell-by-gene matrix in `adata.X`;
- spatial coordinates in `adata.obsm["spatial"]`, or coordinate columns such as `x/y`, `X/Y`, or `center_x/center_y` in `adata.obs`;
- a cell identity column such as `supertype`, `cell_type`, or another annotation used as `phenoLabels`.

In [6]:
def format_gene_names(genes):
    return [gene.upper() if len(gene) == 1 else gene[0].upper() + gene[1:].lower() for gene in genes]

atlas_dirs = [
    '/share/home/liangzhongming/phd_code/530/ProcessedData/MouseBrain/Atlas1',
    '/share/home/liangzhongming/phd_code/530/ProcessedData/MouseBrain/Atlas2',
    '/share/home/liangzhongming/phd_code/530/ProcessedData/MouseBrain/Atlas3',
    '/share/home/liangzhongming/phd_code/530/ProcessedData/MouseBrain/Atlas4'
]

target_files = {
    'well11_cellTypt.h5ad', 
    'C57BL6J-638850.38.h5ad', 
    'Zhuang-ABCA-1.079.h5ad', 
    'S2R1_cellTypt.h5ad'
}

selected_files = {}
for atlas_dir in atlas_dirs:
    h5ad_files = [f for f in os.listdir(atlas_dir) if f.endswith('.h5ad')]
    matched = [f for f in h5ad_files if f in target_files]
    if matched:
        selected_files[atlas_dir] = [os.path.join(atlas_dir, m) for m in matched]

print("The files to be processed：")
for k, v in selected_files.items():
    print(f"{os.path.basename(k)}: {[os.path.basename(f) for f in v]}")

adata_list = []
section_ids = []


for atlas_dir, files in selected_files.items():
    atlas_name = os.path.basename(atlas_dir)
    
    for file_path in files:

        sample = sc.read_h5ad(file_path)
        

        if 'Atlas1' in atlas_name or 'Atlas4' in atlas_name:
            sample.var_names = format_gene_names(sample.var_names)
        elif 'Atlas2' in atlas_name or 'Atlas3' in atlas_name:
            if 'gene_symbol' in sample.var.columns:
                sample.var.index = sample.var['gene_symbol']
                sample.var_names_make_unique()
        

        rename_dict = {
            'class_name': 'class',
            'subclass_name': 'subclass',
            'supertype_name': 'supertype',
            'cluster_name': 'cluster'
        }

        valid_columns = {k: v for k, v in rename_dict.items() if k in sample.obs.columns}
        if valid_columns:
            sample.obs.rename(columns=valid_columns, inplace=True)
            
        for prob_col in ['supertype_bootstrapping_probability', 'subclass_bootstrapping_probability']:
            if prob_col in sample.obs.columns:
                sample = sample[sample.obs[prob_col] >= 0.5].copy()
                

        if 'spatial' not in sample.obsm:
            x_col = next((col for col in ['x', 'X', 'center_x'] if col in sample.obs.columns), None)
            y_col = next((col for col in ['y', 'Y', 'center_y'] if col in sample.obs.columns), None)
            
            if x_col and y_col:
                sample.obsm['spatial'] = sample.obs[[x_col, y_col]].astype(float).values
                
        slice_id = os.path.splitext(os.path.basename(file_path))[0]
        sample.obs['atlas'] = atlas_name
        sample.obs['slice_id'] = slice_id
        
        adata_list.append(sample)
        section_ids.append(f"{atlas_name}_{slice_id}")

The files to be processed：
Atlas1: ['well11_cellTypt.h5ad']
Atlas2: ['C57BL6J-638850.38.h5ad']
Atlas3: ['Zhuang-ABCA-1.079.h5ad']
Atlas4: ['S2R1_cellTypt.h5ad']


## Concatenate slices

We concatenate slices into a single AnnData object.

Important notes:

- `label="sample"` creates `adata_concat.obs["sample"]`, which is required by Squidpy's `library_key`.
- `keys=section_ids` assigns one sample ID to each slice.
- `join="inner"` keeps only genes shared by all slices. This is often safer for cross-atlas or cross-platform integration.
- `index_unique="-"` avoids duplicated cell IDs after concatenation.

In [7]:
adata_concat = sc.concat(
    adata_list, 
    label="sample",
    keys=section_ids,
    join='inner', # inner outer
    index_unique='-' 
)

In [8]:
adata_concat

AnnData object with n_obs × n_vars = 266651 × 52
    obs: 'class', 'subclass', 'supertype', 'cluster', 'cluster_alias', 'atlas', 'slice_id', 'sample'
    obsm: 'spatial'

In [9]:
adata_concat.obs.head()

,class,subclass,supertype,cluster,cluster_alias,atlas,slice_id,sample
0-Atlas1_well11_cellTypt,14 HY Glut,123 DMH Nkx2-4 Glut,0545 DMH Nkx2-4 Glut_2,2227 DMH Nkx2-4 Glut_2,2108.0,Atlas1,well11_cellTypt,Atlas1_well11_cellTypt
1-Atlas1_well11_cellTypt,33 Vascular,329 ABC NN,1186 ABC NN_1,5294 ABC NN_1,5257.0,Atlas1,well11_cellTypt,Atlas1_well11_cellTypt
3-Atlas1_well11_cellTypt,31 OPC-Oligo,327 Oligo NN,1184 MOL NN_4,5285 MOL NN_4,5231.0,Atlas1,well11_cellTypt,Atlas1_well11_cellTypt
4-Atlas1_well11_cellTypt,06 CTX-CGE GABA,049 Lamp5 Gaba,0202 Lamp5 Gaba_4,0722 Lamp5 Gaba_4,713.0,Atlas1,well11_cellTypt,Atlas1_well11_cellTypt
5-Atlas1_well11_cellTypt,30 Astro-Epen,319 Astro-TE NN,1161 Astro-TE NN_1,5220 Astro-TE NN_1,14932.0,Atlas1,well11_cellTypt,Atlas1_well11_cellTypt


## Construct a sample-aware spatial graph with Squidpy

This is the most important step for multi-slice CellNiche.

We use:

```python
library_key="sample"
```

so Squidpy constructs the graph within each sample/slice, instead of connecting cells across unrelated slices.

The default `key_added="spatial"` creates:

```python
adata_concat.obsp["spatial_connectivities"]
adata_concat.obsp["spatial_distances"]
```

CellNiche will later read this graph when the YAML file contains:

```yaml
multi_slice: true
connectivity_key: "spatial"
```

In [10]:
sq.gr.spatial_neighbors(
    adata_concat, 
    library_key='sample',
    coord_type='generic',
    delaunay=True,
    spatial_key='spatial', 
    percentile=99
)

Creating graph using `generic` coordinates and `None` transform and `4` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:01:09)


In [11]:
adata_concat

AnnData object with n_obs × n_vars = 266651 × 52
    obs: 'class', 'subclass', 'supertype', 'cluster', 'cluster_alias', 'atlas', 'slice_id', 'sample'
    uns: 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [23]:
conn = adata_concat.obsp["spatial_connectivities"].tocsr()
row, col = conn.nonzero()

sample_labels = adata_concat.obs["sample"].astype(str).to_numpy()
cross_edge_mask = sample_labels[row] != sample_labels[col]
n_cross_edges = int(cross_edge_mask.sum())

print(f"Number of graph edges: {len(row)}")
print(f"Number of cross-sample edges: {n_cross_edges}")

assert n_cross_edges == 0, (
    "Cross-sample edges were detected. "
    "Please check Squidpy graph construction and library_key='sample'."
)

Number of graph edges: 16020
Number of cross-sample edges: 0


In [12]:
sc.pp.subsample(adata_concat, fraction=0.1, random_state=0, copy=False)

In [13]:
adata_concat.write("/share/home/liangzhongming/phd_code/ModelTest/CellNiche/data/mergedAtlas1234_4slices_subsampled10percent.h5ad")

In [19]:
! cat ../configs/multiSlices.yaml

# ============================================================
#  CellNiche multi-slice training config
# ============================================================

# ------------------------------------------------------------
#  DATA & PRE-PROCESSING
# ------------------------------------------------------------
data_path: "/share/home/liangzhongming/phd_code/ModelTest/CellNiche/data/"
dataset: "mergedAtlas1234_4slices_subsampled10percent"

# Cell identity label in adata.obs.
# Required when embedding_type is "pheno" or "pheno_expr".
# Not required when embedding_type is "expr".
phenoLabels: null

# Ground-truth niche / region label in adata.obs.
# Optional. Set to null if no ground-truth annotation is available.
nicheLabels: null

# Input feature mode:
#   "pheno"      : use one-hot encoded phenoLabels as input features
#   "expr"       : use adata.X as input features
#   "pheno_expr" : use phenoLabels as model input and adata.X for expression-based positive-pair construction
emb

## Train CellNiche

Now we run CellNiche using the YAML config.

Internally, because `multi_slice: true`, CellNiche will:

1. load the `.h5ad` file;
2. read the Squidpy graph from `adata.obsp["spatial_connectivities"]`;
3. use `adata.X"` as cell identity features;
4. train the contrastive model and store the learned embedding in:
   ```python
   adata.obsm["CellNiche"]
   ```

In [20]:
MultiSlices = cn.cli(["--config", "../configs/multiSlices.yaml"])

16:08:48 INFO: Seed: 0
16:08:48 INFO: Using device: cpu


extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)
normalizing counts per cell
    finished (0:00:00)


16:08:48 WARNING: multi_slice=True detected. For multi-slice or multi-sample data, CellNiche expects a precomputed sample-aware spatial graph generated by Squidpy
16:08:48 INFO: Using precomputed Squidpy graph from adata.obsp['spatial_connectivities']: 26665 nodes, 16020 edges.
16:08:48 INFO: Loaded mergedAtlas1234_4slices_subsampled10percent: 26665 nodes, 16020 edges, 32 expression features
16:08:48 INFO: loading_time: 0.31s
16:08:52 INFO: Epoch 1 Step 0001 contrast_loss=6.8173, recon_loss=0.0000
16:08:52 INFO: Epoch 1 Step 0002 contrast_loss=6.6886, recon_loss=0.0000
16:08:52 INFO: Epoch 1 Step 0003 contrast_loss=6.5681, recon_loss=0.0000
16:08:52 INFO: Epoch 1 Step 0004 contrast_loss=6.5057, recon_loss=0.0000
16:08:53 INFO: Epoch 1 Step 0005 contrast_loss=6.4827, recon_loss=0.0000
16:08:53 INFO: Epoch 1 Step 0006 contrast_loss=6.4887, recon_loss=0.0000
16:08:53 INFO: Epoch 1 Step 0007 contrast_loss=6.4574, recon_loss=0.0000
16:08:53 INFO: Epoch 1 Step 0008 contrast_loss=6.4435, reco

In [21]:
MultiSlices

AnnData object with n_obs × n_vars = 26665 × 52
    obs: 'class', 'subclass', 'supertype', 'cluster', 'cluster_alias', 'atlas', 'slice_id', 'sample'
    uns: 'spatial_neighbors'
    obsm: 'spatial', 'CellNiche'
    obsp: 'spatial_connectivities', 'spatial_distances'